In [1]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating project at `~/.julia/dev/drag-formation`


In [2]:
using LinearAlgebra
using SparseArrays
using Convex
using GLPK, MathOptInterface, JuMP

In [3]:
### Constants
const μ_mks = 398600.44 #Earth standard gravitational parameter (km^3/sec^2)
const J2 = 0.00108262668 #Earth J2 gravity coefficient (dimensionless)const 
const Re_mks = 6378.137 #Radius of the Earth (km)
const Ωe_mks = 7.292115e-5 #Rotation rate of Earth (rad/sec)

const Cd = 2.2 #Drag coefficient (unitless)
const AoM_mks = (2+sqrt(2))*3e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg
const AoM0_mks = 1e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg

In [4]:
#Orbital Parameters (roughly ISS)
const a_mks = 6791.0 #semi-major axis (km)
const inc = 54.74 #inclination (deg)
const n_mks = sqrt(μ_mks/a_mks^3) #mean motion (rad/sec)
const P_mks = 2*pi/n_mks; #orbital period (sec) 1.54 hour 5569.440609858254 sec


In [5]:
#Normalize units
const a = 1.0 #distance unit is semi-major axis
const P = 1 #time unit is orbital period
const μ = (2*pi/P)^2 * a^3 #μ_mks*(P_mks^2/a_mks^3)
const n = sqrt(μ/a^3)
const Re = Re_mks/a_mks
const Ωe = Ωe_mks*P_mks;

const AoM = AoM_mks/(a_mks^2)
const AoM0 = AoM0_mks/(a_mks^2);


In [6]:
### Coordinate and unit transformations

function cartesian_to_spherical(x)
    r = sqrt(x[1:3]'*x[1:3]) #range
    ϕ = atan(x[2],x[1]) #azimuth
    θ = atan(x[3],sqrt(x[1:2]'*x[1:2])) #elevation measured up from equatorial plane
    
    return [r; ϕ; θ]
end

function spherical_to_cartesian(s)
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
    x = r*cos(θ)*cos(ϕ)
    y = r*cos(θ)*sin(ϕ)
    z = r*sin(θ)
    
    return [x; y; z]
end

function rtn_frame(x0)
    r = x0[1:3]
    v = x0[4:6]
    n = cross(r,v)
    
    R = r/norm(r)
    N = n/norm(n)
    T = cross(N,R)
    
    return [R T N] #rotation matrix from RTN to Cartesian frame
end

function rtn_to_cartesian(x0,δr)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    r = R*δr + r0 #position in inertial frame
    
    return r
end

function cartesian_to_rtn(x0,r)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    δr = R'*(r - r0) #position in RTN frame
    
    return δr
end

function cartesian_to_aθΩ(x)
    r = x[1:3]
    v = x[4:6]
        
    E = 0.5*v'*v + potential(cartesian_to_spherical(x))
    
    a = -μ/(2*E)
    
    i_x = [1,0,0]
    i_y = [0,1,0]
    i_z = [0,0,1]
    
    h = cross(r,v)
    Ω = atan(h[2]/norm(h[1:2]), h[1]/norm(h[1:2]))
    
    n = cross(i_z,h)/norm(cross(i_z,h))
    if dot(r,i_z) > 0
        θ = acos(dot(n,(r/norm(r))))
    else
        θ = 2*pi- acos(dot(n,(r/norm(r))))
    end
    return [a; θ; Ω]
end

function cartesian_to_ΔaθΩ(x0,x1)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    r1 = x1[1:3]
    v1 = x1[4:6]
    
    E0 = 0.5*v0'*v0 + potential(cartesian_to_spherical(x0))
    E1 = 0.5*v1'*v1 + potential(cartesian_to_spherical(x1))
    
    a0 = -μ/(2*E0)
    a1 = -μ/(2*E1)
    Δa = a1-a0
    

    R = rtn_frame(x0) #rotation from RTN to Cartesian frame    
    rtn1 = R'*r1
    Δθ = atan(rtn1[2]/norm(rtn1[1:2]), rtn1[1]/norm(rtn1[1:2]))
    
    h0 = cross(r0,v0)
    Ω0 = atan(h0[2]/norm(h0[1:2]), h0[1]/norm(h0[1:2]))
    
    h1 = cross(r1,v1)
    Ω1 = atan(h1[2]/norm(h1[1:2]), h1[1]/norm(h0[1:2]))
    
    ΔΩ = Ω1-Ω0
    
    return [Δa; Δθ; ΔΩ]
end

function mks_to_orbital(x)
    #change state from MKS units to orbital units
    
    return [x[1:3]/a_mks; x[4:6]*(P_mks/a_mks)]
end

function orbital_to_mks(x)
    #change state from orbital units to MKS
    
    return [x[1:3]*a_mks; x[4:6]*(a_mks/P_mks)]
end

orbital_to_mks (generic function with 1 method)

In [7]:
function density_mks(alt)
    #This fit is good from 250-500 km altitude
    #Units are km, kg
    a = 106.2
    b = -0.03243
    c = 1.988
    d = -0.01672
    ρ = a*exp(b*alt) + c*exp(d*alt)
end

function atm_density(r)
    #ρ = ρ0*exp((a0-r)/H)
    ρ = (a_mks^3)*density_mks(r*a_mks - Re_mks)
end

atm_density (generic function with 1 method)

In [8]:
#Continuous-Time Linear model
#state is [Δθ; Δa; ΔΩ] where these are differences between satellite 1 and satellite 2
#Can treat drag input as positive part going to sat 2 and negative part going to sat 1
# delta is always with respect to sat 1
function kvalues(a)
    n = sqrt(μ/a^3)
    Fd = -0.5*Cd*atm_density(a)*(AoM)*(n*a)^2*0.965#-15/a_mks
    ȧ = 2*(a^2/μ)*(a)*n*Fd
    k1 = (21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*cosd(inc) #dΩ̇/da
    k2 = -(3.0/2)*sqrt(μ/a^5) #dθ̇/da (dn/da) #(21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*(1-1.5*(sind(inc))^2)#
    k3 = ȧ #dȧ/d(Drag) - 3*sqrt(μ*a)*J2*Re*Re/a^3*sind(inc)^2 
    k4 = k1/k2
    return [k1, k2, k3, k4]
end

kvalues (generic function with 1 method)

In [23]:
#Discrete-time model
num_sat = 1 # number of couple of satellites 


function create_continuous(a, num_sat)
    #Discrete-time model
#     num_sat = 4 # number of couple of satellites 
    k1, k2, k3, k4 = kvalues(a)
    A0 = [0 k2 0; 0 0 0; 0 k1 0]
    A_bar = zeros(3*num_sat,3*num_sat) 
    B_bar = zeros(3*num_sat,num_sat+1) 

    # B_bar[1:3,1:2]=B0
    for k =0:num_sat-1
        A_bar[1+(3*k):3+(3*k),1+(3*k):3+(3*k)] = A0
        B_bar[3*k+2,1] = -k3
        B_bar[3*k+2,2+k] = k3
    end

    return A_bar, B_bar 
end

function c2d(a,dt,num_sat)
    # discretize a continuous time linear system 
    # convert xdot = Ax + Bu 
    # into x_{k+1} = Ad x_k + Bd x_k 
    # where we have dt between k and k +1 
    
    A,B = create_continuous(a, num_sat)
    nx,nu = size(B)
    
    H = exp(dt*[A B;zeros(nu, nx + nu)])
    Ad = H[1:nx, 1:nx]
    Bd = H[1:nx, (nx + 1):(nx + nu)]
    return Ad, Bd 
end


const h =1 #time step
Ad, Bd = c2d(a, h, num_sat)
# A = I + h*A_bar
# B = h*B_bar + 0.5*h*h*A_bar*B_bar;
# B_bar

In [17]:
function sat_dynamics_rk4(x,u,h)
    f1 = satellite_dynamics(x,u)
    f2 = satellite_dynamics(x + 0.5*h*f1,u)
    f3 = satellite_dynamics(x + 0.5*h*f2, u)
    f4 = satellite_dynamics(x + h*f3, u)
    return x + (h/6.0)*(f1 + 2*f2 +2*f3 +f4)
end

sat_dynamics_rk4 (generic function with 1 method)

In [34]:
### Full Orbital Dynamics with J2 and Drag

function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    r1 = x[1:3]
    v1 = x[4:6]
    r2 = x[7:9]
    v2 = x[10:12]
    u1 = u[1]
    u2 = u[2]


    pos_x1 = r1[1]
    pos_y1 = r1[2]
    pos_z1 = r1[3]
    r_norm1 = sqrt(r1[1]^2+r1[2]^2+r1[3]^2)
    
    pos_x2 = r2[1]
    pos_y2 = r2[2]
    pos_z2 = r2[3]
    r_norm2 = sqrt(r2[1]^2+r2[2]^2+r2[3]^2)
    

    g1 = +1.5*J2*μ_mks*Re_mks^2/r_norm1^4*[
        pos_x1/r_norm1*(5*pos_z1^2/r_norm1^2-1);
        pos_y1/r_norm1*(5*pos_z1^2/r_norm1^2-1);
        pos_z1/r_norm1*(5*pos_z1^2/r_norm1^2-3)] - μ_mks/r_norm1^3*r1
    
    g2 = +1.5*J2*μ_mks*Re_mks^2/r_norm2^4*[
        pos_x2/r_norm2*(5*pos_z2^2/r_norm2^2-1);
        pos_y2/r_norm2*(5*pos_z2^2/r_norm2^2-1);
        pos_z2/r_norm2*(5*pos_z2^2/r_norm2^2-3)] - μ_mks/r_norm2^3*r2
    
    v_rel1 = v1 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ1 = density_mks(r_norm1 - Re_mks)
    Fdrag1 = -0.5*(u1*AoM_mks)*Cd*ρ1*(sqrt(v_rel1[1]^2+v_rel1[2]^2+v_rel1[3]^2))*v_rel1

    v_rel2 = v2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ2 = density_mks(r_norm2 - Re_mks)
    Fdrag2 = -0.5*(u2*AoM_mks)*Cd*ρ2*(sqrt(v_rel2[1]^2+v_rel2[2]^2+v_rel2[3]^2))*v_rel2

    v̇1 = g1 + Fdrag1
    v̇2 = g2 + Fdrag2

    return [v1; v̇1; v2; v̇2;]
end


satellite_dynamics (generic function with 1 method)

In [19]:
#Initial and terminal constraints and #Inequality constraints
params = (
    l_12 = 1.0, # number of revolutions
    l_13 = 6.0,
    l_14 = 8.0,
    l_15 = 4.0,
    l_16 = 5.0,
    Δx0 = zeros(size(Ad,1)),
    Δaf = 0, #end at the same altitude
    u_min = 0,
    u_max = 1,
    Δa_min = -10/a_mks /num_sat,
    Δa_max = 10/a_mks /num_sat,) 

k4 = kvalues(a)[4]
params = (; params..., Δθf = 2*pi*[params.l_12,params.l_13,params.l_14,params.l_15,params.l_16],
)

params = (; params..., ΔΩf = k4*(params.Δθf))# Right Ascension Separation


(l_12 = 1.0, l_13 = 6.0, l_14 = 8.0, l_15 = 4.0, l_16 = 5.0, Δx0 = [0.0, 0.0, 0.0], Δaf = 0, u_min = 0, u_max = 1, Δa_min = -0.0014725371815638344, Δa_max = 0.0014725371815638344, Δθf = [6.283185307179586, 37.69911184307752, 50.26548245743669, 25.132741228718345, 31.41592653589793], ΔΩf = [-0.012123811564198254, -0.07274286938518952, -0.09699049251358603, -0.048495246256793016, -0.06061905782099127])

In [20]:
#Discretized Optimal Control Problem Parameters
const Nx = size(Ad,1)     # number of state
const Nu = size(Bd,2)     # number of controls
const Tfinal = 800 # final time (number of orbits)
const Nt = Int(Tfinal/h)+1    # number of time steps
const Nz = Nt*(Nx+Nu) #number of decision variables
thist = Array(range(0,h*(Nt-1), step=h));

#Useful Matrices

# decision variables are stacked z = [u1; x2; u2; x3; u3; ...; xf]
U1 = [sparse(kron(Diagonal(I,Nt), sparse([I zeros(1,(Nu-1+Nx))]))) for i = 1:num_sat] #Matrix that picks out all u of sat 1
U2 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,i) I zeros(1,(Nu-i)+Nx-1)]))) for i = 1:num_sat] #Matrix that picks out all u of sat n
X1 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-1-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x1 = Δθ
X2 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,1) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-2-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x2 = Δa
X3 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,2) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-3-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x3 = ΔΘ


X1F = spzeros(num_sat,(Nu+Nx)*Nt);  #Matrix that picks out x1f = Δθf
X2F = spzeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x2f = Δaf
# X3F = zeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x3f = ΔΘf

for i = num_sat:-1:1
    X1F[num_sat-i+1,end-(i)*3+1] = 1.0
    X2F[num_sat-i+1,end-(i)*3+2] = 1.0
#     X3F[num_sat-i+1,end-(i)*3+3] = 1.0
end

# z = Variable(Nz)

# #Convex Formulation
# objective = 0
# Ac = U1[1] 
# b = zeros(Nt + Nt*num_sat + Nt*num_sat)
# for i in 1:num_sat
#     Ac  = [Ac; U2[i];]
# end

# for i in 1:num_sat
#     Ac  = [Ac; X1[i]]
#     b[Nt + Nt*num_sat + (i-1)*Nt+ 1:Nt + Nt*num_sat + (i)*Nt] .= params.Δθf[i]
# end
# objective += norm(Ac*z-b,1)# + sum(norm(X1[:,:,i]*z - Δθf[i],1) for i in 1:num_sat);

# problem = minimize(objective);


In [21]:
function LP_controller(a,Δx, params)
    z = Variable(Nz)
    # Optimizer
    optimizer = GLPK.Optimizer()
    MOI.set(optimizer, MOI.RawParameter("msg_lev"), GLPK.GLP_MSG_ON)
    
    #Dynamics equality constraints C*z = d
    Ad, Bd = c2d(a, h, num_sat)
    C = sparse([[Bd -I zeros(Nx,(Nt-1)*(Nu+Nx))]; zeros(Nx*(Nt-1),Nu) [kron(Diagonal(I,Nt-1), [Ad Bd]) zeros((Nt-1)*Nx,Nx)] + [zeros((Nt-1)*Nx,Nx) kron(Diagonal(I,Nt-1),[zeros(Nx,Nu) Diagonal(-I,Nx)])]])
    d = [-Ad*Δx; zeros(Nx*(Nt-1))];

    #Convex Formulation
    objective = 0
    Ac = U1[1] 
    b = zeros(Nt + Nt*num_sat + Nt*num_sat)
    for i in 1:num_sat
        Ac  = [Ac; U2[i];]
    end

    for i in 1:num_sat
        Ac  = [Ac; X1[i]]
        b[Nt + Nt*num_sat + (i-1)*Nt+ 1:Nt + Nt*num_sat + (i)*Nt] .= params.Δθf[i]
    end
    objective += norm(Ac*z-b,1)# + sum(norm(X1[:,:,i]*z - Δθf[i],1) for i in 1:num_sat);

    problem = minimize(objective);
    problem.constraints += [C*z == d] #dynamics constraints

    for i = 1:num_sat
        problem.constraints += [X2F[i,:]'*z == params.Δaf] #final altitude constraint
    #     problem.constraints += [X3F[i,:]'*z == ΔΩf] #final altitude constraint
        problem.constraints += [U1[i]*z >= params.u_min; U1[i]*z <= params.u_max] #drag constraints
        problem.constraints += [U2[i]*z >= params.u_min; U2[i]*z <= params.u_max] #drag constraints
        problem.constraints += [X2[i]*z >= params.Δa_min; X2[i]*z <= params.Δa_max]; #altitude constraints
    end

    solve!(problem,optimizer; silent_solver = false)
    
    u1 = zeros(size(U1[1])[1],num_sat) # sat 1 control
    u2 = zeros(size(U2[1])[1],num_sat) # sat n control
    for i = 1:num_sat
        u1[:,i] = U1[i]*z.value;
        u2[:,i] = U2[i]*z.value;
    end
    return [u1[1,:],u2[1,:]]
end

LP_controller (generic function with 1 method)

In [35]:
r0 = [a_mks; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = [r0; v0; r0; v0];

# initialization
u1hist = zeros(size(U1[1])[1],num_sat)
u2hist = zeros(size(U2[1])[1],num_sat)
θhist = zeros(size(X1[1])[1]+1,num_sat)
ahist = zeros(size(X2[1])[1]+1,num_sat)
Ωhist = zeros(size(X3[1])[1]+1,num_sat)

#convert initial condition
x0_orbital = zeros(1,12)
x0_orbital[1:6] = mks_to_orbital(x0[1:6])
x0_orbital[7:12] = mks_to_orbital(x0[7:12])

x0c = cartesian_to_ΔaθΩ(mks_to_orbital(x0[1:6]),mks_to_orbital(x0[7:12]))

# a1 = cartesian_to_aθΩ(x0_orbital[1:6])[1] 
# a2 = cartesian_to_aθΩ(x0_orbital[7:12])[1] 

# find first controller
u0 = LP_controller(a, x0c, params)
println(x0c)
println(u0)
println(zeros(size(Ad,1)))
#store first step
u1hist[1,:] .= u0[1]
u2hist[1,:] .= u0[2]
θhist[1,:] .= x0c[1]
ahist[1,:] .= x0c[2]
Ωhist[1,:] .= x0c[3]

xk = x0
xc = x0c
for k = 1:(Nt-1)
    println(k)
#     uk = LP_controller(a,xc,params)  # find control
    uk = [0.0, 1.0]
    xk = sat_dynamics_rk4(xk,uk,h)  #simulate dynamics
    println("xk ",xk, " uk ", uk)
#     xk_orbital = zeros(1,12)
#     xk_orbital[1:6] = mks_to_orbital(xk[1:6]) #transform dynamics output first sat into normalized coordinates
#     xk_orbital[7:12] = mks_to_orbital(xk[7:12])  #transform dynamics output second sat into normalized coordinates
    xc = cartesian_to_ΔaθΩ(mks_to_orbital(xk[1:6]),mks_to_orbital(xk[7:12])) # tranform output into difference for control
    println("xc ",xc[1], " uk ", uk)
#     a1 = cartesian_to_aθΩ(xk[1:6])[1] # find new a to update control dynamics
#     a2 = cartesian_to_aθΩ(xk[7:12])[1] #  find new a to update control dynamics
#     println("a1 ", a1, " a2 ", a2)
    u1hist[k,:] .= uk[1]
    u2hist[k,:] .= uk[2]
    θhist[k,:] .= xc[1]
    ahist[k,:] .= xc[2]
    Ωhist[k,:] .= xc[3]
end

      0: obj =   0.000000000e+00 inf =   2.520e+03 (401)
   2987: obj =   2.333175045e+03 inf =   0.000e+00 (0) 12
*  3257: obj =   2.332792693e+03 inf =   2.065e-11 (0) 3
[0.0, 0.0, 0.0]
[[0.0], [1.0]]
[0.0, 0.0, 0.0]
1
xk [6790.995672252181, 4.422768856326417, 6.25575634978679, -0.00865549471322069, 4.422766977307001, 6.255753684416589, 6790.995672252182, 4.422768854952047, 6.25575634784282, -0.008655494710531005, 4.422766974558262, 6.2557536805286516] uk [0.0, 1.0]
xc -1.247757452915721e-9 uk [0.0, 1.0]
2
xk [6790.982689014269, 8.845532075595498, 12.511504703464276, -0.017310978336869592, 4.4227585217228205, 6.25574169025523, 6790.982689014277, 8.845532070098022, 12.511504695688405, -0.017310978326110886, 4.422758516225349, 6.255741682479365] uk [0.0, 1.0]
xc -2.495514239697627e-9 uk [0.0, 1.0]
3
xk [6790.9610503029, 13.268284020757203, 18.76723706493354, -0.02596643978138989, 4.422744429094682, 6.2557217000035985, 6790.961050302924, 13.268284008387894, 18.767237047437845, -0.025966

xk [6785.699201705146, 154.75666456282886, 218.894384546269, -0.30286314194303887, 4.4193175533513855, 6.250860712877928, 6785.699201743566, 154.7566628796249, 218.894382165474, -0.3028631386513908, 4.419317457191133, 6.250860576864895] uk [0.0, 1.0]
xc -4.3672868432764744e-8 uk [0.0, 1.0]
36
xk [6785.39201427577, 159.17588255496122, 225.14510403265925, -0.3115116512143221, 4.419117492169894, 6.25057692830734, 6785.3920143175765, 159.17588077422465, 225.14510151391008, -0.31151164773208995, 4.419117393264926, 6.25057678841208] uk [0.0, 1.0]
xc -4.4920746233856335e-8 uk [0.0, 1.0]
37
xk [6785.076178535754, 163.5948976697249, 231.39553573975465, -0.3201597613772849, 4.418911798657809, 6.250285154350472, 6785.07617858114, 163.5948957887111, 231.39553307916924, -0.3201597576991225, 4.418911697008355, 6.25028501057331] uk [0.0, 1.0]
xc -4.616863158446449e-8 uk [0.0, 1.0]
38
xk [6784.751694889745, 168.01370427492185, 237.6456716783574, -0.32880746135232, 4.418700473081397, 6.249985391386043,

xk [6769.805125175656, 309.2717360150132, 437.44607205607684, -0.6052510147861475, 4.4089662995553285, 6.236177589481938, 6769.80512548246, 309.2717292869813, 437.44606253968493, -0.6052510016580753, 4.408966107507693, 6.236177317843036] uk [0.0, 1.0]
xc -8.735398249193338e-8 uk [0.0, 1.0]
71
xk [6769.195560119532, 313.68050428949465, 443.6819687497907, -0.6138789673043321, 4.408569312918545, 6.235614469557067, 6769.195560439653, 313.6804973680487, 443.6819589598271, -0.613878953800011, 4.4085691181379545, 6.235614194052625] uk [0.0, 1.0]
xc -8.860221822892811e-8 uk [0.0, 1.0]
72
xk [6768.577367503203, 318.0888727679577, 449.9172983385352, -0.6225061333549889, 4.408166707604679, 6.235043379665796, 6768.577367837018, 318.0888656503648, 449.9172882711347, -0.6225061194691687, 4.408166510091581, 6.235043100296448] uk [0.0, 1.0]
xc -8.985046617837611e-8 uk [0.0, 1.0]
73
xk [6767.9505481186625, 322.49683583198515, 456.1520528527136, -0.6311325018867645, 4.407758484134824, 6.234464320549293,

xk [6743.34271173025, 463.3040042836332, 655.3129103014289, -0.9066891347831941, 4.391732445278594, 6.211731653994977, 6743.342712762676, 463.3039891633964, 655.3128889148524, -0.9066891053889065, 4.391732157883326, 6.211731247496018] uk [0.0, 1.0]
xc -1.3105159846205083e-7 uk [0.0, 1.0]
106
xk [6742.431725569006, 467.69544055390963, 661.5242218368841, -0.9152829932012121, 4.391139162523572, 6.210890093843853, 6742.431726631104, 467.69542514492065, 661.5242000418893, -0.9152829632494053, 4.391138872414588, 6.2108896835066485] uk [0.0, 1.0]
xc -1.3230042728018532e-7 uk [0.0, 1.0]
107
xk [6741.51214613469, 472.08628074330693, 667.7346878431556, -0.9238756790916337, 4.390540283648888, 6.210040595810505, 6741.512147227021, 472.08626504285235, 667.7346656359049, -0.9238756485772143, 4.390539990826843, 6.2100401816359865] uk [0.0, 1.0]
xc -1.3354927497211122e-7 uk [0.0, 1.0]
108
xk [6740.583974605333, 476.47651925609233, 673.9443003829115, -0.9324671814495941, 4.389935809429495, 6.2091831609

xk [6706.353484445452, 616.6130246615156, 872.1538356605175, -1.2067045869726456, 4.367643308127851, 6.177561761163146, 6706.353486882678, 616.612997825093, 872.1537977023632, -1.206704535072087, 4.367642926179672, 6.177561220931666] uk [0.0, 1.0]
xc -1.7477397662446492e-7 uk [0.0, 1.0]
141
xk [6705.142506587578, 620.9802741141305, 878.3308387480366, -1.2152508702439762, 4.366854669566779, 6.176443098221865, 6705.142509077071, 620.9802468944159, 878.3308002477503, -1.215250817611122, 4.366854284931174, 6.17644255418944] uk [0.0, 1.0]
xc -1.760236343351096e-7 uk [0.0, 1.0]
142
xk [6703.922983223848, 625.3467321457489, 884.5067192258965, -1.2237955968632033, 4.366060466305645, 6.175316542087408, 6703.922985766341, 625.3467045400554, 884.5066801796778, -1.2237955434931755, 4.366060078983463, 6.175315994255244] uk [0.0, 1.0]
xc -1.7727331758088383e-7 uk [0.0, 1.0]
143
xk [6702.694915916383, 629.712393192184, 890.6814692016338, -1.2323387558900718, 4.365260699371834, 6.174182094221055, 6702

...

xk [6534.9766094767165, 1065.5602549767982, 1507.119889127051, -2.08519940171425, 4.256040340989559, 6.0192577507822165, 6534.976622136303, 1065.5601740641653, 1507.1197746846674, -2.0851992490001843, 4.256039684976022, 6.019256822951113] uk [0.0, 1.0]
xc -3.049188963544225e-7 uk [0.0, 1.0]
245
xk [6532.887246728656, 1069.8156154874466, 1513.1381825792116, -2.0935256485472866, 4.254679776878643, 6.017327872172842, 6532.887259541553, 1069.815533917513, 1513.1380672071766, -2.0935254946388313, 4.254679118290903, 6.0173269407014045] uk [0.0, 1.0]
xc -3.0617241209007773e-7 uk [0.0, 1.0]
246
xk [6530.789559073117, 1074.069612723991, 1519.154542309081, -2.101849214895693, 4.2533137930751606, 6.0153903066171495, 6530.789572040522, 1074.0695304941833, 1519.1544260037551, -2.101849059788921, 4.2533131319145125, 6.015389371507214] uk [0.0, 1.0]
xc -3.074259742330554e-7 uk [0.0, 1.0]
247
xk [6528.683549195906, 1078.3222412676207, 1525.168960630967, -2.1101700901163847, 4.251942391343648, 6.013445

xk [6456.913727929688, 1213.6513534713479, 1716.5594272120243, -2.3749536870190333, 4.205207792910664, 5.947155210163874, 6456.913746692066, 1213.6512480305623, 1716.5592780786387, -2.3749534902391285, 4.205207047589338, 5.947154156040542] uk [0.0, 1.0]
xc -3.488213133895357e-7 uk [0.0, 1.0]
280
xk [6454.534660928084, 1217.8557871315, 1722.5054843731837, -2.383179808606109, 4.203658634925813, 5.944957846378748, 6454.534679887904, 1217.8556809441293, 1722.505334183888, -2.3831796105011906, 4.203657887077275, 5.944956788681725] uk [0.0, 1.0]
xc -3.5007659726637996e-7 uk [0.0, 1.0]
281
xk [6452.147369329394, 1222.0586689565976, 1728.4493403737033, -2.3914028794393203, 4.202104123135605, 5.942752889358967, 6452.147388487982, 1222.0585620201152, 1728.4491891249245, -2.3914026800058696, 4.2021033727612425, 5.942751828090222] uk [0.0, 1.0]
xc -3.513319348780186e-7 uk [0.0, 1.0]
282
xk [6449.751856189626, 1226.259993593838, 1734.3909876217747, -2.399622889010204, 4.200544259546839, 5.940540341

xk [6368.7732979439825, 1359.8482631075217, 1923.312201756055, -2.6609841851040024, 4.147815242502338, 5.865748694477542, 6368.773324428607, 1359.8481300421709, 1923.3120135530805, -2.6609839399360733, 4.147814409566702, 5.865747516463569] uk [0.0, 1.0]
xc -3.927896472522008e-7 uk [0.0, 1.0]
315
xk [6366.108256914541, 1363.995211138916, 1929.1767203921368, -2.6690973053044105, 4.146079940182757, 5.863287329495416, 6366.108283645054, 1363.9950772393904, 1929.1765310093965, -2.6690970586957365, 4.146079104769561, 5.863286147978162] uk [0.0, 1.0]
xc -3.940469361918275e-7 uk [0.0, 1.0]
316
xk [6363.435104472048, 1368.1404212280538, 1935.0387739191956, -2.6772070094823417, 4.144339358363636, 5.860818476963403, 6363.435131449891, 1368.140286491877, 1935.0385833531873, -2.6772067614298467, 4.144338520474327, 5.860817291944916] uk [0.0, 1.0]
xc -3.95304285971676e-7 uk [0.0, 1.0]
317
xk [6360.753844037703, 1372.2838880965583, 1940.8983548512779, -2.68531328728089, 4.142593499290402, 5.858342140

xk [6273.6323516018165, 1499.838030925627, 2121.278264272658, -2.934852123535611, 4.085866913600354, 5.777881324908847, 6273.632387273514, 1499.8378681175311, 2121.278034005208, -2.9348518277357565, 4.085865997243762, 5.777880028940325] uk [0.0, 1.0]
xc -4.3557269790994724e-7 uk [0.0, 1.0]
349
xk [6270.693503568305, 1503.9229415199507, 2127.0547891658125, -2.942843316753245, 4.083953408277336, 5.775167232174685, 6270.693539536571, 1503.922777794284, 2127.0545576006775, -2.9428430194155113, 4.083952489492763, 5.775165932773172] uk [0.0, 1.0]
xc -4.368321608883008e-7 uk [0.0, 1.0]
350
xk [6267.74666622348, 1508.005936009054, 2132.8285962791397, -2.9508307444610997, 4.0820347035715026, 5.772445765839342, 6267.746702489854, 1508.0057713633894, 2132.8283634128875, -2.950830445582867, 4.082033782360415, 5.772444463006915] uk [0.0, 1.0]
xc -4.38091689813902e-7 uk [0.0, 1.0]
351
xk [6264.79184333795, 1512.0870091947904, 2138.5996782407965, -2.9588143964653804, 4.0801108019562164, 5.76971692942

xk [6166.044240780226, 1641.6359290797172, 2321.791495319804, -3.2122369571320073, 4.01581712927271, 5.67852430347039, 6166.044287764023, 1641.6357327220148, 2321.7912176049404, -3.2122366060380068, 4.015816128807066, 5.678522888580747] uk [0.0, 1.0]
xc -4.796951535990246e-7 uk [0.0, 1.0]
384
xk [6162.828076795901, 1645.650699650448, 2327.4685352201113, -3.220090325814836, 4.013723160482796, 5.675554289342018, 6162.8281241316035, 1645.6505022910915, 2327.468256088677, -3.2200899730995145, 4.013722157640251, 5.675552871091783] uk [0.0, 1.0]
xc -4.809570843811173e-7 uk [0.0, 1.0]
385
xk [6159.604061501666, 1649.66337369772, 2333.1426014835242, -3.2279395752844104, 4.011624082805736, 5.6725770303218495, 6159.6041091908955, 1649.6631753343333, 2333.14232093216, -3.2279392209456366, 4.011623077587736, 5.672575608713073] uk [0.0, 1.0]
xc -4.82219089326108e-7 uk [0.0, 1.0]
386
xk [6156.37219902174, 1653.673946113998, 2338.8136868670726, -3.235784695530819, 4.009519898945161, 5.66959253025516,

In [ ]:
# res = LP_controller(params.Δx0, params)
# u1hist = zeros(size(U1[1])[1],num_sat) # sat 1 control
# u2hist = zeros(size(U2[1])[1],num_sat) # sat n control
# θhist = zeros(size(X1[1])[1]+1,num_sat)
# ahist = zeros(size(X2[1])[1]+1,num_sat)
# Ωhist = zeros(size(X3[1])[1]+1,num_sat)
# for i = 1:num_sat
#     u1hist[:,i] = U1[i]*res;
#     u2hist[:,i] = U2[i]*res;
#     θhist[:,i] = [params.Δx0[1]; X1[i]*res];
#     ahist[:,i] = [params.Δx0[2]; X2[i]*res];
#     Ωhist[:,i] = [params.Δx0[3]; X3[i]*res];
# end


In [25]:
using Plots
plotlyjs()

mylabel = String[]
mylabel1 = String[]
mylabel2 = String[]
for i = 1:num_sat
    append!(mylabel,["Sat. 1-"*string(i+1)])
    append!(mylabel1,["Sat. 1"])
    append!(mylabel2,["Sat. "*string(i+1)])
end

plot(thist,u1hist,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(thist,u2hist,label=permutedims(mylabel2))
plot!(xlabel="Time, s", ylabel = "Control")

┌ Info: Precompiling PlotlyJS [f0f68f2c-4968-5e81-91da-67840de0976a]
└ @ Base loading.jl:1423


WebIO._IJuliaInit()

In [26]:
plot(thist[1:end],broadcast.(rad2deg,θhist[2:end,:]),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "True Anomaly Difference, deg")

In [36]:
plot(thist,a_mks*ahist[2:end,:], label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Semi-major axis Difference, km")


In [28]:
plot(thist,broadcast.(rad2deg,Ωhist[2:end,:]),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "RAAN Difference, deg")

In [29]:
plot(thist,a_mks*Ωhist[2:end,:],label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [ ]:
### Full Orbital Dynamics with J2 and Drag

function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r = x[1:3]
    v = x[4:6]
    r_noJ2 = x[7:9]
    v_noJ2 = x[10:12]

    pos_x = r[1]
    pos_y = r[2]
    pos_z = r[3]
    r_norm = sqrt(r[1]^2+r[2]^2+r[3]^2)
    r_norm_noJ2 = sqrt(r_noJ2[1]^2+r_noJ2[2]^2+r_noJ2[3]^2)
    
    g = +1.5*J2*μ_mks*Re_mks^2/r_norm^4*[
        pos_x/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_y/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_z/r_norm*(5*pos_z^2/r_norm^2-3)] - μ_mks/r_norm^3*r
    
    
    v_rel = v #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ = density_mks(r_norm - Re_mks)
    Fdrag = -0.5*(u*AoM_mks)*Cd*ρ*(sqrt(v_rel[1]^2+v_rel[2]^2+v_rel[3]^2))*v_rel

    v_rel_noJ2 = v_noJ2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ_noJ2 = density_mks(r_norm_noJ2 - Re_mks)
    Fdrag_noJ2 = -0.5*(u*AoM_mks)*Cd*ρ_noJ2*(sqrt(v_rel_noJ2[1]^2+v_rel_noJ2[2]^2+v_rel_noJ2[3]^2))*v_rel_noJ2

    v̇ = g + Fdrag
    v̇_noJ2 = -μ_mks/(r_norm_noJ2)^3 * r_noJ2 +  Fdrag_noJ2
    return [v; v̇; v_noJ2; v̇_noJ2]
end


In [ ]:
#Initial conditions in Cartesian Coordinates
# r0 = [a; 0; 0]
# v0 = [0; cosd(inc)*sqrt(μ/a); sind(inc)*sqrt(μ/a)]
# x0 = [r0; v0; r0; v0];

# r1 = [a; 0; 0]
# v1 = [0; cosd(inc)*sqrt(μ/(a)); sind(inc)*sqrt(μ/(a))]
# x1 = [r1; v1; r1; v1];

# tspan = (0.0, Tfinal+50)
# Nt_t = Tfinal+150

r0 = [a_mks; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = [r0; v0; r0; v0];

r1 = [a_mks; 0; 0]
v1 = [0; cosd(inc)*sqrt(μ_mks/(a_mks)); sind(inc)*sqrt(μ_mks/(a_mks))]
x1 = [r1; v1; r1; v1];

step_t = 60*10 # save results every 10 mins
tspan = (0.0, Int(round((Tfinal)*P_mks/step_t)*step_t))
Nt_t = Int(round((Tfinal)*P_mks/step_t))-1#/100

In [ ]:
using Interpolations
#Full Orbit Mechanics Simulation and save results
# Nt = 150
# Nt = Nt*3
temp = LinRange(tspan[1],tspan[2],Nt_t)
x0_soln = zeros(6*num_sat,Nt_t)
x1_soln = zeros(6*num_sat,Nt_t)

x0_noJ2_soln = zeros(6*num_sat,Nt_t)
x1_noJ2_soln = zeros(6*num_sat,Nt_t)

Δa_soln = zeros(num_sat,Nt_t)
Δθ_soln = zeros(num_sat,Nt_t)
ΔΩ_soln = zeros(num_sat,Nt_t)

r_soln1 = zeros(num_sat,Nt_t)
ϕ_soln1 = zeros(num_sat,Nt_t)
θ_soln1 = zeros(num_sat,Nt_t)
u_soln1 = zeros(num_sat,Nt_t)

r_soln2 = zeros(num_sat,Nt_t)
ϕ_soln2 = zeros(num_sat,Nt_t)
θ_soln2 = zeros(num_sat,Nt_t)
u_soln2 = zeros(num_sat,Nt_t)

a_soln1 = zeros(num_sat,Nt_t)
ν_soln1 = zeros(num_sat,Nt_t)
Ω_soln1 = zeros(num_sat,Nt_t)

a_soln2 = zeros(num_sat,Nt_t)
ν_soln2 = zeros(num_sat,Nt_t)
Ω_soln2 = zeros(num_sat,Nt_t)

r_soln1_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln1_noJ2 = zeros(num_sat,Nt_t)
θ_soln1_noJ2 = zeros(num_sat,Nt_t)
u_soln1_noJ2 = zeros(num_sat,Nt_t)

r_soln2_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln2_noJ2 = zeros(num_sat,Nt_t)
θ_soln2_noJ2 = zeros(num_sat,Nt_t)
u_soln2_noJ2 = zeros(num_sat,Nt_t)

h_traj = 1#step_t#/P_mks
thist_c = thist.*P_mks
for i = 1:num_sat
    u1interp = LinearInterpolation(thist_c, u1hist[:,i])
    u2interp = LinearInterpolation(thist_c, u2hist[:,i])

    x_hist1 = zeros(length(x0),Int(tspan[2]/h_traj))
    x_hist2 = zeros(length(x1),Int(tspan[2]/h_traj))
    x_hist1[:,1] = x0 
    x_hist2[:,1] = x1 
    temp = LinRange(tspan[1],tspan[2],Nt_t)
    k = 0 
    save_index = 0
    for t = tspan[1]:h_traj:tspan[2]-step_t# temp# 1:100:tspan[2]-1
        k += 1
        #Stored Controller
        if t < tspan[2]
            u1 = u1interp(t) 
            u2 = u2interp(t) 
        else
            u1 = 0;#u_soln1[i,k]
            u2 = 0;#u_soln2[i,k]
        end

        x_hist1[:,k+1] .= sat_dynamics_rk4(x_hist1[:,k],u1,h_traj)
        x_hist2[:,k+1] .= sat_dynamics_rk4(x_hist2[:,k],u2,h_traj)

        # save results
        if k%step_t == 0
            save_index += 1
            u_soln1[i,save_index] = u1
            u_soln2[i,save_index] = u2
            x0_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[1:6,k]
            x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[7:12,k]
        
            x1_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[1:6,k]
            x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[7:12,k]
    
            rϕθ1_noJ2 = cartesian_to_spherical(x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2_noJ2 = cartesian_to_spherical(x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ1 = cartesian_to_spherical(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2 = cartesian_to_spherical(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            ΔaθΩ = cartesian_to_ΔaθΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]), mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ1 = cartesian_to_aθΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ2 = cartesian_to_aθΩ(mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
    
    
            Δa_soln[i,save_index] = ΔaθΩ[1]
            Δθ_soln[i,save_index] = ΔaθΩ[2]
            ΔΩ_soln[i,save_index] = ΔaθΩ[3]

            a_soln1[i,save_index] = aθΩ1[1]
            ν_soln1[i,save_index] = aθΩ1[2]
            Ω_soln1[i,save_index] = aθΩ1[3]
        
            a_soln2[i,save_index] = aθΩ2[1]
            ν_soln2[i,save_index] = aθΩ2[2]
            Ω_soln2[i,save_index] = aθΩ2[3]
        
            r_soln1_noJ2[i,save_index] = rϕθ1_noJ2[1]
            ϕ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[2]
            θ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[3]
    
            r_soln2_noJ2[i,save_index] = rϕθ2_noJ2[1]
            ϕ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[2]
            θ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[3]
        
            r_soln1[i,save_index] = rϕθ1[1]
            ϕ_soln1[i,save_index] = rϕθ1[2]
            θ_soln1[i,save_index] = rϕθ1[3]
    
            r_soln2[i,save_index] = rϕθ2[1]
            ϕ_soln2[i,save_index] = rϕθ2[2]
            θ_soln2[i,save_index] = rϕθ2[3]
        end

    end
end
tsamp = LinRange(tspan[1],tspan[2],Nt_t);


In [ ]:
using Plots
# tsamp = LinRange(tspan[1],tspan[2],Nt)
plot(tsamp[1:30:end]./(60*60*24),x0_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[3,1:30:end],color=:green)

plot!(tsamp[1:30:end]./(60*60*24),x1_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[3,1:30:end],color=:green)

# plot(tsamp,x0_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x0_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x0_noJ2_soln[3,:],color=:green)

# plot!(tsamp,x1_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x1_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x1_noJ2_soln[3,:],color=:green)
# x1_soln[3,:]

In [ ]:
plot(tsamp[1:30:end]./(60*60*24),broadcast.(rad2deg,ν_soln1[1,1:30:end]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln1[1,:]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln2[1,:]),color=:blue)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

# length(Ω_soln1[1,:][Ω_soln1[1,:] .>3.])

In [ ]:
tsamp = LinRange(tspan[1],tspan[2],Nt_t)
plot(tsamp[1:30:end]./(60*60*24),Δa_soln[1:30:end]',label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "Altitude Difference, km")


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Δθ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "True Anomaly Difference, deg")

In [ ]:
plot(tsamp./(60*40*24),broadcast.(rad2deg,ΔΩ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "RAAN Difference, deg", ylim = [-10,5])

In [ ]:
plot(tsamp,a_mks*ΔΩ_soln',label=permutedims(mylabel), ylim = [-100*1e1,10*1e2])
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [ ]:
plot(tsamp./(60*60*24),u_soln1',color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),u_soln2',label=permutedims(mylabel2))
plot!(xlabel="Time, days", ylabel = "Control")


In [ ]:
plot(tsamp./(60*40*24),r_soln1_noJ2'.-Re_mks,color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*40*24),r_soln2_noJ2'.-Re_mks,label=permutedims(mylabel2))
for i=1:num_sat
    println(r_soln1_noJ2[1]-Re_mks," ",params.Δa_max*a_mks*i)
    plot!([tsamp[1]./(60*40*24), tsamp[end]./(60*40*24)],[r_soln1_noJ2[1]-Re_mks-params.Δa_max*a_mks*i, r_soln1_noJ2[1]-Re_mks[1]-params.Δa_max*a_mks*i],color=:red, linewidth=2,label="Limit")
end
plot!(xlabel= "Time, days",ylabel= "Altitude, km")


In [ ]:
plot(tsamp[1:30:end]./(60*40*24),ϕ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),ϕ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Azimuth, rad")

In [ ]:
plot(tsamp[1:30:end]./(60*40*24),θ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),θ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Elevation, rad")

In [ ]:
# Final RAAN
broadcast.(rad2deg,ΔΩ_soln[:,end]')

println("Targeted Δθf ",  broadcast.(rad2deg, params.Δθf[1:num_sat]), " deg" )
println("Reached Δθf ",  broadcast.(rad2deg, Δθ_soln[:,end]), " deg" )


println("Targeted ΔΩ ",  broadcast.(rad2deg, params.ΔΩf[1:num_sat]), " deg" )
println("Reached ΔΩ ",  broadcast.(rad2deg, ΔΩ_soln[:,end]), " deg" )
println("Final Error ΔΩ ", broadcast.(rad2deg, (params.ΔΩf[1:num_sat] .- ΔΩ_soln[:,end])), " deg" )
println("Final Azimut Sat 1 ", broadcast.(rad2deg, ϕ_soln1[:,end]), " deg")
println("Final Azimut Sat Coupled ", broadcast.(rad2deg, ϕ_soln2[:,end]), " deg")

println("Final Elevation Sat 1 ", broadcast.(rad2deg, θ_soln1[:,end]), " deg")
println("Final Elevation Sat Coupled ", broadcast.(rad2deg, θ_soln2[:,end]), " deg")

In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
using Plots

green_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.9, 0.5, 0.8))
red_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.9, 0.5, 0.5, 0.8))
blue_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.5, 0.9, 0.3))


vis = Visualizer()
open(vis)
IJuliaCell(vis)
setprop!(vis["/Background"], "top_color", colorant"transparent") # makes it cool night style

# create earth
origin = [0, 0, 0] # Might be different if from COM or from edge
len = 2.0
radius = Re
earth = Sphere(Point(0.0, 0.0, 0.0), radius)


# Add Object to visualizer
setobject!(vis["earth"], earth, blue_material) 



size_spacecraft = Vec(0.05,0.05,0.05)
size_plate = Vec(size_spacecraft[1]*3,0.01,0.04)

origin_spacecraft = Vec(x0_soln[1,1]/Re_mks-size_spacecraft[1]/2,x0_soln[2,1]/Re_mks-size_spacecraft[2]/2,x0_soln[3,1]/Re_mks-size_spacecraft[3]/2)
origin_plate = Vec(x0_soln[1,1]/Re_mks-size_plate[1]/2,x0_soln[2,1]/Re_mks-size_plate[2]/2,x0_soln[3,1]/Re_mks-size_plate[3]/2)


cubeshape = HyperRectangle(origin_spacecraft,size_spacecraft)
plateshape =HyperRectangle(origin_plate,size_plate)
center = Sphere(Point(origin_spacecraft[1],origin_spacecraft[2],origin_spacecraft[3]), 0.01)

colors = []
for i =1: num_sat+1
    append!(colors,rand(Float64, (3)))
    spacecraft_material=MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(colors[(i-1)*3+1], colors[(i-1)*3+2], colors[(i-1)*3+3], 0.5))
    name_sat = "satellite"*string(i)
    name_plate = "plate"*string(i)
    setobject!(vis[name_sat], cubeshape, spacecraft_material) 
    setobject!(vis[name_sat][name_plate], plateshape, spacecraft_material)
end

sleep(5)
for i = 1:length(x0_soln[1,:])-1
    for j = 1 : num_sat
        if j == 1
#             positions_1 = [x0_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x0_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x0_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
            positions_1 = [x0_soln[1+(6*j-6),i]/Re_mks-origin_spacecraft[1],x0_soln[2+(6*j-6),i]/Re_mks-origin_spacecraft[2],x0_soln[3+(6*j-6),i]/Re_mks-origin_spacecraft[3]]
            rotations_1 = i*2*pi/length(x0_soln[1+(6*j-6),:])
            position_1 = Translation(positions_1)
#             rotation_1 = LinearMap(RotZ(rotations_1))
            name_sat = "satellite"*string(j)
            settransform!(vis[name_sat], position_1)
#             settransform!(vis[name_sat], compose(position_1, rotation_1))
        end 
#         positions_2 = [x1_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x1_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x1_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
        positions_2 = [x1_soln[1+(6*j-6),i]/Re_mks-origin_spacecraft[1],x1_soln[2+(6*j-6),i]/Re_mks-origin_spacecraft[2],x1_soln[3+(6*j-6),i]/Re_mks-origin_spacecraft[3]]
#         rotations_2 = i*2*pi/length(x1_soln[1+(6*j-6),:])
        position_2 = Translation(positions_2)
#         rotation_2 = LinearMap(RotZ(rotations_2))
        name_sat = "satellite"*string(j+1)
        settransform!(vis[name_sat], position_2)
#         settransform!(vis[name_sat], compose(position_2, rotation_2))
    end
        #     settransform!(vis["plate1"], position)
    sleep(0.2)
end


In [ ]:
# using NaNStatistics
# window_size = length(tsamp)/1.0
# z1 = movmean(θ_soln1, window_size)
# z2 = movmean(θ_soln2, window_size)
# plot(tsamp,θ_soln1,linestyle=:dot)
# plot!(tsamp,θ_soln2,linestyle=:dot)
# plot!(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)

In [ ]:
# window_size = length(tsamp)/100.0
# z1 = movmean(Ω_soln1, window_size)
# z2 = movmean(Ω_soln2, window_size)
# # plot(tsamp,Ω_soln1,linestyle=:dot)
# # plot!(tsamp,Ω_soln2,linestyle=:dot)
# plot(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)
# plot!(ylim= [-1,1])

In [ ]:
k1/k2

In [ ]:
(k1/k2)*2*pi

In [ ]:
((7.0/2)*(Re*Re/(a*a))*J2*cosd(inc))*2*pi